# 03a · GasWindTunnel: R3 encoder features

**Purpose.** Inside each fold, train the shared 1D CNN encoder + a temporary linear layer on the training labels only, drop the linear layer, freeze the encoder and extract features. Fit the 4 heads on both tasks.

**Inputs**
- GasWindTunnel `X_signals.npy` (600, 72, 1500) + `meta.csv`

**Outputs**
- Per sensor: (1, 1500) → (16, 750) → (32, 375) → GAP (32,)
- F_enc: (600, 72, 32) → (600, 2304)
- `results/03a_gwt_encoder.csv`

## Colab setup
Run this first on Google Colab (Runtime → Change runtime type → **GPU**). It clones the repo, downloads the
processed data from the shared Drive folder into `data/processed/`, and moves into this notebook's folder so
that the rest of the notebook runs unchanged. On a laptop it does nothing.

In [1]:
import glob
import os
import shutil
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ
REPO = "/content/gas-repr-guide"
REPO_URL = "https://github.com/YasserAlbogami/gas-repr-guide.git"
DATA_URL = "https://drive.google.com/drive/folders/1rNIcZkHDFhH2dMVq8TgNWt_0IH2yw2_I"   # processed data (Yasser)
DATA_DIRS = ["gas_wind_tunnel", "everyday_scents"]

if IN_COLAB:
    # 1. code (src/). Tried in this order:
    #    a) clone the GitHub repo (works when the repo is public)
    #    b) clone with a GitHub token saved in Colab Secrets (key icon on the left) as GITHUB_TOKEN
    #    c) src.zip uploaded to /content (Files panel on the left)
    if not os.path.exists(f"{REPO}/src"):
        env = dict(os.environ, GIT_TERMINAL_PROMPT="0")           # fail instead of asking for a password
        ok = subprocess.run(["git", "clone", "-q", REPO_URL, REPO], env=env).returncode == 0
        if not ok:
            token = None
            try:
                from google.colab import userdata
                token = userdata.get("GITHUB_TOKEN")
            except Exception:
                pass
            if token:
                url = REPO_URL.replace("https://", f"https://{token}@")
                ok = subprocess.run(["git", "clone", "-q", url, REPO], env=env).returncode == 0
        if not ok and os.path.exists("/content/src.zip"):
            os.makedirs(f"{REPO}/notebooks/03_encoder", exist_ok=True)
            shutil.unpack_archive("/content/src.zip", REPO)
            ok = True
        if not ok:
            raise RuntimeError("Could not get the code: the repo is private. Make it public, or add GITHUB_TOKEN "
                               "in Colab Secrets, or upload src.zip to /content, then run this cell again.")
    elif os.path.exists(f"{REPO}/.git"):
        subprocess.run(["git", "-C", REPO, "pull", "-q"], check=False)
    os.makedirs(f"{REPO}/results", exist_ok=True)

    # 2. processed data (only downloaded once per Colab session)
    processed = f"{REPO}/data/processed"
    if not all(os.path.exists(f"{processed}/{d}/X_signals.npy") for d in DATA_DIRS):
        import gdown
        tmp = "/content/processed_download"
        gdown.download_folder(DATA_URL, output=tmp, quiet=False)
        for d in DATA_DIRS:
            found = glob.glob(f"{tmp}/**/{d}", recursive=True)
            assert found, f"folder {d} not found in the Drive download"
            shutil.copytree(found[0], f"{processed}/{d}", dirs_exist_ok=True)
    for d in DATA_DIRS:
        files = sorted(os.listdir(f"{processed}/{d}"))
        print(d, files)
        for f in ["X_signals.npy", "meta.csv", "info.json"]:
            assert f in files, f"{d}/{f} is missing"

    # 3. run the rest of the notebook from its own folder, as on a laptop (so ../.. = project root)
    os.chdir(f"{REPO}/notebooks/03_encoder")

    import torch
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
          else "NONE, switch on Runtime > Change runtime type > GPU")

Retrieving folder contents


Retrieving folder 1YSf-59uCQxHrruY705o36CXtRQ0XS2Fr everyday_scents
Processing file 1hdYRGs6ijl-mYorzn2esfZaWM_Tnd7Ey baseline_level.npy
Processing file 1oPfagu_BzcC2ZvL2SX7bNG8_UDIGygZv info.json
Processing file 1skq8t0lHKIjaht2QFEVxxVZCJf7Wqfvf info.json
Processing file 1KHEy1xcDVFGO3ipc7kExOHk7rUtNCKkr meta.csv
Processing file 1LGP9oFexIJepxC2ZEzbDUAgw2xAYoBeL X_signals.npy
Retrieving folder 1WOzILzMT3EEu4nsar-gg1IH0GNSse4x5 gas_wind_tunnel
Processing file 1Wh2umVgpw82YCugVFMEBM8g6ropQYpvr baseline_level.npy
Processing file 1TRpQKW_t8W5liucNyAt84_I7K2At7QB2 info.json
Processing file 1eX05vgphoRl020zxcypuktJ2OEKstp6O meta.csv
Processing file 128GyQQJ_BsxwQI3fZDSBSEeHhQvtDlGE X_signals.npy


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1hdYRGs6ijl-mYorzn2esfZaWM_Tnd7Ey
To: /content/processed_download/everyday_scents/baseline_level.npy
100%|██████████| 6.46k/6.46k [00:00<00:00, 11.4MB/s]
Downloading...
From: https://drive.google.com/uc?id=1oPfagu_BzcC2ZvL2SX7bNG8_UDIGygZv
To: /content/processed_download/everyday_scents/info.json
100%|██████████| 297/297 [00:00<00:00, 1.09MB/s]
Downloading...
From: https://drive.google.com/uc?id=1skq8t0lHKIjaht2QFEVxxVZCJf7Wqfvf
To: /content/processed_download/everyday_scents/info.json
100%|██████████| 297/297 [00:00<00:00, 1.41MB/s]
Downloading...
From: https://drive.google.com/uc?id=1KHEy1xcDVFGO3ipc7kExOHk7rUtNCKkr
To: /content/processed_download/everyday_scents/meta.csv
100%|██████████| 26.7k/26.7k [00:00<00:00, 43.7MB/s]
Downloading...
From: https://drive.google.com/uc?id=1LGP9oFexIJepxC2ZEzbDUAgw2xAYoBeL
To: /content/processe

gas_wind_tunnel ['.gitkeep', 'X_signals.npy', 'baseline_level.npy', 'info.json', 'meta.csv']
everyday_scents ['.gitkeep', 'X_signals.npy', 'baseline_level.npy', 'info.json', 'meta.csv']
GPU: NVIDIA A100-SXM4-40GB


In [2]:
import sys
sys.path.append("../..")          # project root, so that `import src` works

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path("../..")
from src.cv import TASKS, load_baseline_level, load_processed, precomputed, run_cv, save_results, summarize
from src.features.encoder import encoder_builder
# about 30 encoder trainings (5 folds x 3 seeds x 2 tasks): this notebook is the slow one
DATASET = "gwt"
REP = "R3_encoder"
NAME = "03a_gwt_encoder"          # results/03a_gwt_encoder.csv

X, meta, info = load_processed(DATASET)    # X: (N, C, T) baseline corrected
print(X.shape, meta["y_multi"].value_counts().sort_index().tolist())

(600, 72, 1500) [60, 60, 60, 60, 60, 60, 60, 60, 60, 60]


## R3: shared 1D CNN encoder, 32 features per sensor
The encoder is trained **inside each fold** on the training fold only (encoder + temporary linear layer,
early stopping on an inner validation split). The linear layer is then dropped, the encoder is frozen and
encodes the training and test curves; the 4 heads are fitted on those features exactly as for R1 and R2.

Shape check on 2 samples (untrained network):

In [3]:
import torch
from src.features.encoder import SensorEncoder

with torch.no_grad():
    z = SensorEncoder(32)(torch.tensor(X[:2]))                     # (2, C * 32)
print(z.shape, "-> full feature matrix", (len(X), z.shape[1]))
assert (len(X), z.shape[1]) == (600, 2304)

torch.Size([2, 2304]) -> full feature matrix (600, 2304)


## Cross validation: 4 heads × 5 folds × 3 seeds, both tasks
Everything that is fitted (scaler, heads, hyperparameters, encoder) sees the training fold only.

In [4]:
scores, preds = [], []
for task in TASKS:                                   # "multi", then "binary"
    s, p = run_cv(X, meta, DATASET, task, REP, encoder_builder(n_out=32))
    scores.append(s)
    preds.append(p)
scores, preds = pd.concat(scores, ignore_index=True), pd.concat(preds, ignore_index=True)
save_results(scores, preds, NAME)
print("saved results/" + NAME + ".csv")

gwt multi R3_encoder seed 0 fold 0: logreg 0.958  svm 0.966  rf 0.975  mlp 0.958
gwt multi R3_encoder seed 0 fold 1: logreg 0.992  svm 0.983  rf 0.992  mlp 0.967
gwt multi R3_encoder seed 0 fold 2: logreg 1.000  svm 1.000  rf 1.000  mlp 0.983
gwt multi R3_encoder seed 0 fold 3: logreg 0.967  svm 0.959  rf 0.983  mlp 0.967
gwt multi R3_encoder seed 0 fold 4: logreg 0.992  svm 0.984  rf 0.983  mlp 0.966
gwt multi R3_encoder seed 1 fold 0: logreg 0.983  svm 0.959  rf 1.000  mlp 0.950
gwt multi R3_encoder seed 1 fold 1: logreg 0.967  svm 0.959  rf 0.967  mlp 0.966
gwt multi R3_encoder seed 1 fold 2: logreg 0.992  svm 1.000  rf 1.000  mlp 1.000
gwt multi R3_encoder seed 1 fold 3: logreg 0.975  svm 0.960  rf 0.992  mlp 0.975
gwt multi R3_encoder seed 1 fold 4: logreg 0.975  svm 0.951  rf 0.992  mlp 0.960
gwt multi R3_encoder seed 2 fold 0: logreg 0.992  svm 0.992  rf 0.983  mlp 0.984
gwt multi R3_encoder seed 2 fold 1: logreg 0.975  svm 0.959  rf 0.992  mlp 0.959
gwt multi R3_encoder seed 2 

## Results: mean ± std over 3 seeds × 5 folds

## Colab: save the results
Colab deletes everything when the session ends. This copies the results csv files to your Google Drive
(`MyDrive/gas-repr-guide/results/`) and also downloads them to your computer. On a laptop it does nothing.

In [7]:
if IN_COLAB:
    out_files = sorted(glob.glob(f"{REPO}/results/{NAME}*.csv"))
    print(out_files)
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        out_dir = "/content/drive/MyDrive/gas-repr-guide/results"
        os.makedirs(out_dir, exist_ok=True)
        for f in out_files:
            shutil.copy(f, out_dir)
        print("copied to", out_dir)
    except Exception as e:
        print("Drive copy failed:", e)
    from google.colab import files
    for f in out_files:
        files.download(f)

['/content/gas-repr-guide/results/03a_gwt_encoder.csv', '/content/gas-repr-guide/results/03a_gwt_encoder_preds.csv']
Drive copy failed: mount failed


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
print(4)

4


In [9]:
print("macro F1"); display(summarize(scores))
print("balanced accuracy"); display(summarize(scores, "balanced_acc"))

macro F1


head                              logreg            mlp             rf  \
dataset task   rep                                                       
gwt     binary R3_encoder  0.980 ± 0.018  0.961 ± 0.036  0.997 ± 0.008   
        multi  R3_encoder  0.981 ± 0.012  0.970 ± 0.013  0.987 ± 0.010   

head                                 svm  
dataset task   rep                        
gwt     binary R3_encoder  0.980 ± 0.029  
        multi  R3_encoder  0.971 ± 0.018

balanced accuracy


head                              logreg            mlp             rf  \
dataset task   rep                                                       
gwt     binary R3_encoder  0.986 ± 0.025  0.965 ± 0.046  0.994 ± 0.015   
        multi  R3_encoder  0.981 ± 0.012  0.970 ± 0.013  0.987 ± 0.010   

head                                 svm  
dataset task   rep                        
gwt     binary R3_encoder  0.967 ± 0.045  
        multi  R3_encoder  0.971 ± 0.018

print("ROC AUC, binary task"); summarize(scores[scores["task"] == "binary"], "roc_auc")

In [10]:
print("ROC AUC, binary task"); summarize(scores[scores["task"] == "binary"], "roc_auc")

ROC AUC, binary task


,,head,logreg,mlp,rf,svm
dataset,task,rep,,,,
gwt,binary,R3_encoder,0.992 ± 0.014,0.977 ± 0.034,1.000 ± 0.000,0.990 ± 0.022


In [11]:
if IN_COLAB:
    out_files = sorted(glob.glob(f"{REPO}/results/{NAME}*.csv"))
    print(out_files)
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        out_dir = "/content/drive/MyDrive/gas-repr-guide/results"
        os.makedirs(out_dir, exist_ok=True)
        for f in out_files:
            shutil.copy(f, out_dir)
        print("copied to", out_dir)
    except Exception as e:
        print("Drive copy failed:", e)
    from google.colab import files
    for f in out_files:
        files.download(f)


['/content/gas-repr-guide/results/03a_gwt_encoder.csv', '/content/gas-repr-guide/results/03a_gwt_encoder_preds.csv']
Mounted at /content/drive
copied to /content/drive/MyDrive/gas-repr-guide/results


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>